# Recording and replaying NanoVer sessions

This notebook demonstrates how to record a NanoVer session and later provide playback of that recording on a NanoVer server. NanoVer recordings capture the simulated system and shared network state as seen by a client, and therefore capture users' physical movements and interactions alongside the frame by frame state of the system. This level of recording is a useful alternative to typical trajectories when investigating the behavior of iMD users or other NanoVer specific tools in conjuction with iMD simulations.

## Simulation setup

We set up a simple simulation server, in the usual way, just to demonstrate the recording features.

In [ ]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

omm_sim = OpenMMSimulation.from_bundle_path("../systems/openmm/17-ala.openmm.zip")
imd_runner = OmniRunner.with_basic_server(omm_sim, port=0, name="recording and replay example")

## Recording on the fly

We can use the utilities to expose start/stop recording user commands to the XR client, that allow you to make recordings on the fly from virtual reality:

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.use_recording_commands()

Next we simply connect to the server from the NanoVer iMD-XR client and use the commands menu to trigger recordings, which will be saved alongside this notebook:

<video controls src="../figures/recording_with_checkpoints.webm">

## Recording full sessions

We can also attach a recorder to the server directly, that will capture everything until the server closes:

In [ ]:
from nanover.websocket.record import record_from_runner

recorder = record_from_runner(imd_runner, "recording.nanover.zip")

When you're done, you must ensure to close at least one of the recorder or the server so that it can terminate the recording cleanly:

In [ ]:
recorder.close()

## Recording playback

Finally, we load the simulation file as a NanoVer compatible simulation, add it to the list of simulations on the server, and switch to it:

In [ ]:
from nanover.recording import PlaybackSimulation

playback_sim = PlaybackSimulation.from_path(path="recording.nanover.zip")

In [ ]:
imd_runner.add_simulation(playback_sim)
imd_runner.load(1)